# 04 — Baseline Anomaly Detection

## 1. Objective

Establish simple, interpretable anomaly-detection baselines for the SWaT dataset before applying machine-learning and deep-learning methods.

The baselines provide reference performance against which the more complex models developed in later notebooks can be evaluated.

## 2. Baseline Methods

This notebook evaluates three baseline approaches:

1. **Rule-Based Detection**
   - Detect observations that violate predefined sensor-value conditions.
   - Provides an interpretable operational baseline.

2. **Rolling Statistical Detection**
   - Detect observations whose sensor values deviate substantially from their recent temporal behaviour.
   - Uses rolling statistics while preserving chronological order.

3. **Simple Statistical Threshold**
   - Detect observations using thresholds derived from normal training data.
   - Provides a simple distribution-based reference without using attack labels during threshold construction.

## 3. Data Handling Principles

- Raw files in `data/raw` are not modified.
- Project-relative path discovery is used.
- Temporal ordering is preserved.
- The normal and attack datasets are not artificially concatenated to create a continuous time series.
- The missing-data block identified in Notebook 02 is not filled using blanket forward/backward imputation.
- Thresholds and statistical parameters are derived only from data available during model development.
- Attack labels are reserved for evaluation and are not used to construct detection thresholds.
- The final evaluation period must remain independent of threshold selection.

## 4. Relationship to Previous Notebooks

Notebook 02 established the data-quality and temporal-integrity constraints that must be respected during modelling.

Notebook 03 established exploratory characteristics of the sensor variables, including:

- substantial missingness in seven normal sensor variables;
- constant behaviour in several sensors;
- differences between normal and attack sensor distributions;
- structured relationships among sensor variables; and
- temporal and multivariate characteristics relevant to anomaly detection.

These findings guide the baseline design but do not determine anomaly thresholds by themselves.

## 5. Evaluation Principle

The baselines are calibrated and validated on chronologically separated portions of the normal data. Attack labels are not used to construct detection rules or select thresholds.

This notebook establishes baseline behaviour on the normal validation period. Attack-labelled evaluation, including confusion-matrix and classification-metric analysis, is deferred to the next notebook, where the same calibration/validation split is preserved.

Performance will be interpreted as a reference point rather than as evidence that a baseline represents the underlying anomaly mechanism.

## 6. Expected Outcome

The notebook will establish reproducible baseline detection behaviour and performance on the normal validation period, which can be compared with the LSTM autoencoder, traditional machine-learning methods, and deep-learning methods developed in later notebooks.

Attack-labelled evaluation is outside the scope of this notebook and is addressed in the next stage.

## 7. Baseline Calibration and Evaluation Data

Baseline detection rules and statistical thresholds must be established without using attack labels or future evaluation observations.

The normal data contain a substantial missing-data block and a repeated temporal segment identified during data-quality analysis. Therefore, baseline calibration will use a clearly defined normal-data period that is suitable for parameter estimation and does not require blanket imputation.

The normal validation period is chronologically disjoint from the normal calibration period. The attack dataset, however, overlaps the calibration and validation time windows (Notebook 02, Section 17) even though it shares zero exact timestamps with them. Attack-side evaluation must therefore be treated as a separate, temporally overlapping evaluation, not as a temporally independent hold-out.

No threshold will be selected by maximizing performance on attack labels. Attack labels will be used only after the detection procedure has been fixed, for performance evaluation.

Chronological separation between normal calibration and normal validation data is required to prevent information leakage and to ensure that the normal-side baseline results represent a valid out-of-sample reference. The time-window overlap with the attack data is documented above and is reflected in the scope of Sections 5 and 6.

## 8. Mathematical Foundation

Let `x_t,j` denote the value of sensor `j` at time `t`.

For a baseline detector, an anomaly score `s_t` is calculated for each observation. A threshold `τ` is then used to classify the observation:

**Anomaly rule:**

`s_t > τ  →  anomaly`

`s_t ≤ τ  →  normal`

where:

- `s_t` is the anomaly score at time `t`;
- `τ` is the detection threshold.

The three baseline methods differ primarily in how the anomaly score and threshold are defined.

### 8.1 Rule-Based Detection

A rule-based detector applies an explicit condition to a sensor value.

For a sensor with acceptable limits `L_j` and `U_j`:

**Anomaly condition:**

`x_t,j < L_j` or `x_t,j > U_j`

**Normal condition:**

`L_j ≤ x_t,j ≤ U_j`

This method is highly interpretable because each detection decision can be directly traced to a predefined sensor condition.

### 8.2 Rolling Statistical Detection

For a rolling window containing the previous `w` observations, the rolling mean is:

**Rolling mean:**

`μ_t,w = (1 / w) × Σ(x_i)`

where the summation is over the previous `w` observations before time `t`.

The rolling standard deviation is:

**Rolling standard deviation:**

`σ_t,w = sqrt[Σ(x_i - μ_t,w)² / (w - 1)]`

The standardized deviation of the current observation is:

**Rolling z-score:**

`z_t = |x_t - μ_t,w| / σ_t,w`

An observation is classified as anomalous when:

`z_t > τ`

Only previous observations are used to calculate the rolling mean and standard deviation. This prevents the current observation from influencing its own reference distribution and preserves the causal temporal structure of the detector.

### 8.3 Simple Statistical Threshold

For a sensor with `n` normal calibration observations, the sample mean is:

**Sample mean:**

`x̄ = (1 / n) × Σ(x_i)`

The sample standard deviation is:

**Sample standard deviation:**

`s = sqrt[Σ(x_i - x̄)² / (n - 1)]`

The standardized deviation of an observation is:

**Statistical z-score:**

`z_t = |x_t - x̄| / s`

The observation is flagged as anomalous when:

`z_t > τ`

The mean and standard deviation must be estimated from the designated calibration data only.

### 8.4 Interpretation

These baselines make different assumptions:

- Rule-based detection relies on predefined acceptable operating conditions.
- Rolling statistical detection models deviation from recent temporal behaviour.
- Simple statistical thresholding models deviation from the overall normal calibration distribution.

The baselines are intentionally simple. Their purpose is to establish interpretable reference performance before introducing models capable of learning more complex temporal and multivariate patterns.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw" / "normal.csv").is_file()
        and (directory / "data" / "raw" / "attack.csv").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/normal.csv' and 'data/raw/attack.csv'."
    )

DATA_ROOT = PROJECT_ROOT / "data" / "raw"

normal_path = DATA_ROOT / "normal.csv"
attack_path = DATA_ROOT / "attack.csv"

# print(f"Project root: {PROJECT_ROOT}")
print(f"Normal dataset found: {normal_path.is_file()}")

print(f"Attack dataset found: {attack_path.is_file()}")

Normal dataset found: True
Attack dataset found: True


In [4]:
import warnings

# Suppress pandas UserWarnings
warnings.filterwarnings('ignore', category=UserWarning)

normal = pd.read_csv(normal_path, low_memory=False)
attack = pd.read_csv(attack_path, low_memory=False)

normal.columns = normal.columns.str.strip()
attack.columns = attack.columns.str.strip()

TIMESTAMP_COL = "Timestamp"
LABEL_COL = "Normal/Attack"

for name, dataset in [("Normal", normal), ("Attack", attack)]:
    if TIMESTAMP_COL not in dataset.columns:
        raise KeyError(f"{name} dataset is missing '{TIMESTAMP_COL}'.")

    if LABEL_COL not in dataset.columns:
        raise KeyError(f"{name} dataset is missing '{LABEL_COL}'.")

    dataset[TIMESTAMP_COL] = pd.to_datetime(
        dataset[TIMESTAMP_COL],
        errors="coerce"
    )

normal_features = set(normal.columns) - {TIMESTAMP_COL, LABEL_COL}
attack_features = set(attack.columns) - {TIMESTAMP_COL, LABEL_COL}

if normal_features != attack_features:
    raise ValueError("Normal and attack datasets have different sensor columns.")

feature_cols = [
    column
    for column in normal.columns
    if column not in (TIMESTAMP_COL, LABEL_COL)
]

print(f"Normal dataset shape: {normal.shape}")
print(f"Attack dataset shape: {attack.shape}")
print(f"Number of sensor features: {len(feature_cols)}")
print(f"Invalid normal timestamps: {normal[TIMESTAMP_COL].isna().sum()}")
print(f"Invalid attack timestamps: {attack[TIMESTAMP_COL].isna().sum()}")
print(f"Normal labels: {normal[LABEL_COL].value_counts(dropna=False).to_dict()}")
print(f"Attack labels: {attack[LABEL_COL].value_counts(dropna=False).to_dict()}")
print(
    "Normal missing sensor cells:",
    int(normal[feature_cols].isna().sum().sum())
)
print(
    "Attack missing sensor cells:",
    int(attack[feature_cols].isna().sum().sum())
)

Normal dataset shape: (1387098, 53)
Attack dataset shape: (54621, 53)
Number of sensor features: 51
Invalid normal timestamps: 5
Invalid attack timestamps: 0
Normal labels: {'Normal': 1387098}
Attack labels: {'Attack': 54621}
Normal missing sensor cells: 6942600
Attack missing sensor cells: 0


In [5]:
normal_timestamps = normal[TIMESTAMP_COL]

valid_timestamp_mask = normal_timestamps.notna()
complete_sensor_mask = normal[feature_cols].notna().all(axis=1)

eligible_mask = valid_timestamp_mask & complete_sensor_mask

eligible_normal = normal.loc[eligible_mask].copy()

print(f"Total normal observations: {len(normal):,}")
print(f"Normal observations with valid timestamps: {valid_timestamp_mask.sum():,}")
print(f"Normal observations with complete sensor values: {complete_sensor_mask.sum():,}")
print(f"Normal observations meeting both conditions: {eligible_mask.sum():,}")
print(f"Normal observations excluded by these conditions: {(~eligible_mask).sum():,}")

print("\nEligible observations by original row position:")

eligible_positions = np.flatnonzero(eligible_mask.to_numpy())

if len(eligible_positions):
    segment_starts = np.r_[0, np.flatnonzero(np.diff(eligible_positions) > 1) + 1]
    segment_ends = np.r_[segment_starts[1:], len(eligible_positions)]

    for start, end in zip(segment_starts, segment_ends):
        segment = eligible_normal.iloc[start:end]

        print(
            f"Rows {eligible_positions[start] + 1:,}–"
            f"{eligible_positions[end - 1] + 1:,}: "
            f"{len(segment):,} observations | "
            f"{segment[TIMESTAMP_COL].min()} to "
            f"{segment[TIMESTAMP_COL].max()}"
        )

Total normal observations: 1,387,098
Normal observations with valid timestamps: 1,387,093
Normal observations with complete sensor values: 395,298
Normal observations meeting both conditions: 395,298
Normal observations excluded by these conditions: 991,800

Eligible observations by original row position:
Rows 1–395,298: 395,298 observations | 2015-12-28 10:00:00 to 2016-01-02 14:59:59


In [6]:
# Examine temporal integrity of the eligible normal observations.

timestamps = eligible_normal[TIMESTAMP_COL]

time_differences = timestamps.diff().dt.total_seconds()

print("TEMPORAL INTEGRITY CHECK")
print("-" * 40)

print(f"Observations: {len(eligible_normal):,}")
print(f"Timestamps are non-decreasing: {timestamps.is_monotonic_increasing}")
print(f"Duplicate timestamps: {int(timestamps.duplicated().sum()):,}")
print(f"Negative time intervals: {int((time_differences < 0).sum()):,}")
print(f"Zero-second intervals: {int((time_differences == 0).sum()):,}")
print(f"Missing intervals: {int(time_differences.isna().sum()):,}")

positive_intervals = time_differences[time_differences > 0]

if not positive_intervals.empty:
    print("\nPositive time-interval statistics (seconds):")
    print(positive_intervals.describe().to_string())

    print("\nMost frequent positive intervals (seconds):")
    print(positive_intervals.value_counts().head(10).to_string())

TEMPORAL INTEGRITY CHECK
----------------------------------------
Observations: 395,298
Timestamps are non-decreasing: True
Duplicate timestamps: 0
Negative time intervals: 0
Zero-second intervals: 0
Missing intervals: 1

Positive time-interval statistics (seconds):
count    395297.000000
mean          1.138382
std          57.418260
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max       35901.000000

Most frequent positive intervals (seconds):
Timestamp
1.0      395261
722.0         3
941.0         1
444.0         1
384.0         1
391.0         1
197.0         1
430.0         1
965.0         1
234.0         1


## 9. Temporal Calibration and Evaluation Split

The eligible normal segment identified above contains 395,298 observations with valid timestamps and complete sensor values.

Because anomaly detection is a temporal problem, the observations are partitioned chronologically rather than randomly.

The normal observations are divided as follows:

- **80% calibration period:** used to estimate the parameters required by the baseline detectors.
- **20% validation period:** used to assess baseline behaviour and, where required, select detection thresholds without using attack labels.
- **Attack dataset:** reserved for final evaluation of detection performance.

The chronological order of the normal observations is preserved throughout the split.

The attack labels are therefore not used to construct the baseline rules, estimate normal-data statistics, or select detection thresholds.

The attack dataset is evaluated only after the detection procedures have been fixed.

This separation provides a simple baseline against which the more complex models developed in later notebooks can be compared while reducing the risk of evaluation leakage.

In [8]:
n_eligible = len(eligible_normal)
calibration_end = int(n_eligible * 0.80)

normal_calibration = eligible_normal.iloc[:calibration_end].copy()
normal_validation = eligible_normal.iloc[calibration_end:].copy()

print("TEMPORAL PARTITION")
print("-" * 40)

print(
    f"Calibration observations: {len(normal_calibration):,} "
    f"({len(normal_calibration) / n_eligible:.1%})"
)

print(
    f"Validation observations: {len(normal_validation):,} "
    f"({len(normal_validation) / n_eligible:.1%})"
)

print("\nCalibration period:")
print(
    f"{normal_calibration[TIMESTAMP_COL].min()} "
    f"to {normal_calibration[TIMESTAMP_COL].max()}"
)

print("\nValidation period:")
print(
    f"{normal_validation[TIMESTAMP_COL].min()} "
    f"to {normal_validation[TIMESTAMP_COL].max()}"
)

print("\nChronological separation:")
print(
    normal_calibration[TIMESTAMP_COL].max()
    < normal_validation[TIMESTAMP_COL].min()
)

TEMPORAL PARTITION
----------------------------------------
Calibration observations: 316,238 (80.0%)
Validation observations: 79,060 (20.0%)

Calibration period:
2015-12-28 10:00:00 to 2016-01-01 15:51:19

Validation period:
2016-01-01 15:51:20 to 2016-01-02 14:59:59

Chronological separation:
True


## 10. Baseline 1 — Rule-Based Detection

### 10.1 Approach

A rule-based detector flags observations whose sensor values fall outside empirically defined normal ranges.

Because verified engineering operating limits have not been established for this analysis, no safety limits or process constraints are invented.

Instead, the lower and upper reference limits are derived from the normal calibration period.

For each sensor \(j\), the empirical lower and upper limits are defined using selected quantiles of the calibration distribution:

- Lower limit: calibration lower quantile
- Upper limit: calibration upper quantile

An observation is flagged when its sensor value falls below the lower limit or above the upper limit.

The attack labels are not used to determine these limits.

### 10.2 Mathematical Definition

For sensor \(j\), let \(Q_{\alpha,j}\) and \(Q_{1-\alpha,j}\) represent the lower and upper quantiles calculated from the normal calibration data.

The rule is:

`Anomaly = 1 if x_t,j < Q_α,j or x_t,j > Q_(1-α),j`

Otherwise:

`Anomaly = 0`

The resulting detector is therefore based entirely on the empirical distribution of the normal calibration observations.

### 10.3 Interpretation

This baseline answers a simple question:

> How effectively can unusual individual sensor values be identified using only empirical normal operating ranges?

The method is intentionally univariate and interpretable. It does not model relationships between sensors or temporal dependencies.

Its purpose is to provide a transparent reference point for the more advanced anomaly-detection methods developed later.

In [10]:
from IPython.display import display

# Empirical limits for the rule-based baseline.
# Limits are calculated from normal calibration data only.

LOWER_QUANTILE = 0.01
UPPER_QUANTILE = 0.99

rule_limits = pd.DataFrame({
    "lower_limit": normal_calibration[feature_cols].quantile(LOWER_QUANTILE),
    "upper_limit": normal_calibration[feature_cols].quantile(UPPER_QUANTILE),
})

print("Rule-based empirical limits")
print("-" * 40)
print(f"Lower quantile: {LOWER_QUANTILE:.2%}")
print(f"Upper quantile: {UPPER_QUANTILE:.2%}")
print(f"Features with limits: {len(rule_limits)}")

display(rule_limits)

Rule-based empirical limits
----------------------------------------
Lower quantile: 1.00%
Upper quantile: 99.00%
Features with limits: 51


,lower_limit,upper_limit
FIT101,0.000000,2.669827
LIT101,314.571600,814.887300
MV101,1.000000,2.000000
P101,1.000000,2.000000
P102,1.000000,1.000000
AIT201,175.948500,263.426100
AIT202,8.355102,8.692514
AIT203,291.668800,356.036900
FIT201,0.000000,2.473200
MV201,1.000000,2.000000


In [11]:
# Apply calibration-derived rules to the normal validation period.

validation_values = normal_validation[feature_cols]

lower_violations = validation_values.lt(rule_limits["lower_limit"], axis="columns")
upper_violations = validation_values.gt(rule_limits["upper_limit"], axis="columns")

validation_rule_flags = lower_violations | upper_violations

normal_validation_rule_score = validation_rule_flags.sum(axis=1)
normal_validation_rule_anomaly = normal_validation_rule_score > 0

print("RULE-BASED DETECTION — NORMAL VALIDATION")
print("-" * 50)

print(f"Validation observations: {len(normal_validation):,}")
print(
    f"Observations flagged as anomalous: "
    f"{normal_validation_rule_anomaly.sum():,}"
)
print(
    f"Validation anomaly rate: "
    f"{normal_validation_rule_anomaly.mean():.4%}"
)
print(
    f"Minimum violated sensors in an observation: "
    f"{normal_validation_rule_score.min()}"
)
print(
    f"Maximum violated sensors in an observation: "
    f"{normal_validation_rule_score.max()}"
)
print(
    f"Mean violated sensors per observation: "
    f"{normal_validation_rule_score.mean():.4f}"
)

print("\nMost frequently violated sensors:")
display(
    validation_rule_flags.sum()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("violation_count")
)

RULE-BASED DETECTION — NORMAL VALIDATION
--------------------------------------------------
Validation observations: 79,060
Observations flagged as anomalous: 73,295
Validation anomaly rate: 92.7081%
Minimum violated sensors in an observation: 0
Maximum violated sensors in an observation: 16
Mean violated sensors per observation: 1.9914

Most frequently violated sensors:


,violation_count
AIT201,63020
AIT402,21867
AIT203,13506
AIT202,11921
AIT504,10927
AIT502,10043
DPIT301,4923
FIT201,3231
LIT301,3176
FIT504,2986


## 10.4 Rule-Based Validation Findings

The empirical rule-based detector was constructed using the 1st and 99th percentiles of the normal calibration period and then applied unchanged to the normal validation period.

The validation results were:

- Validation observations: 79,060
- Observations flagged as anomalous: 73,295
- Validation anomaly rate: 92.7081%
- Maximum number of simultaneously violated sensors: 16
- Mean number of violated sensors per observation: 1.9914

The detector therefore produces a very high false-alarm rate on the normal validation period.

The most frequently violated sensors were AIT201, AIT402, AIT203, AIT202, AIT504, AIT502, and DPIT301.

This result indicates that the empirical 1st/99th-percentile limits estimated from the calibration period do not generalize well to the subsequent normal validation period.

The result is retained as observed rather than corrected through repeated threshold adjustment. In particular, attack observations and attack labels have not been used to modify the rules.

### Interpretation

The high validation anomaly rate suggests that normal sensor behaviour is not adequately represented by fixed marginal limits derived from the earlier calibration period.

Possible contributors include changes in normal operating conditions and the temporal/non-stationary behaviour of the process. The result does not by itself establish the cause of the distribution shift.

This limitation is important because a useful anomaly detector must distinguish unusual behaviour from legitimate changes in normal operating conditions.

The rule-based detector will therefore be treated as a simple reference baseline rather than as a final detection method.

## 11. Baseline 2 — Rolling Statistical Detection

### 11.1 Objective

Rolling statistical detection identifies observations that deviate substantially from recent sensor behaviour. Unlike the rule-based baseline, which uses fixed limits estimated from the calibration period, this method compares each observation with a reference calculated from preceding observations.

### 11.2 Statistical Method

For a sensor with observations x₁, x₂, ..., xₜ, the rolling mean and standard deviation are calculated from the previous w observations, excluding the current observation.

**Rolling mean**

μₜ = (xₜ₋w + xₜ₋w₊₁ + ... + xₜ₋₁) / w

**Rolling standard deviation**

σₜ = √[Σ(xᵢ − μₜ)² / (w − 1)]

where the summation covers the previous w observations.

**Rolling Z-score**

zₜ = |xₜ − μₜ| / σₜ

The rolling Z-score measures how far the current sensor reading deviates from its recent historical mean in units of the historical standard deviation.

An observation is flagged when its rolling Z-score exceeds a predefined threshold.

If the rolling standard deviation is zero, the Z-score cannot be calculated using the expression above. The implementation must handle this case explicitly to avoid division by zero.

### 11.3 Implementation Principles

- Calculate reference statistics using preceding observations only, excluding the observation being assessed.
- Use the normal calibration period to establish the rolling reference and assess the method on normal validation observations.
- Preserve chronological order; do not randomly sample or shuffle observations.
- Handle the initial rolling-window warm-up period explicitly.
- Handle zero rolling standard deviation without producing infinite or undefined scores.
- Do not invent engineering limits or use attack labels to select thresholds.
- Record the resulting validation anomaly rate and the sensors contributing most frequently to detections.

### 11.4 Interpretation

A rolling reference can adapt to gradual changes in sensor behaviour, but it can also miss persistent deviations if the reference gradually follows them. Its performance depends on the window length, threshold, sensor characteristics, and temporal behaviour of the process.

The results will be interpreted as a baseline comparison, not as proof of a confirmed attack.

In [12]:
# Rolling statistical detection: define the baseline parameters.

ROLLING_WINDOW = 60
ROLLING_Z_THRESHOLD = 3.0

print("ROLLING STATISTICAL DETECTION — PARAMETERS")
print("-" * 50)
print(f"Rolling window: {ROLLING_WINDOW} preceding observations")
print(f"Z-score threshold: {ROLLING_Z_THRESHOLD}")
print("Reference data: normal calibration period")
print("Assessment data: normal validation period")
print("Current observation excluded from rolling reference")

ROLLING STATISTICAL DETECTION — PARAMETERS
--------------------------------------------------
Rolling window: 60 preceding observations
Z-score threshold: 3.0
Reference data: normal calibration period
Assessment data: normal validation period
Current observation excluded from rolling reference


In [14]:
# Calculate rolling Z-scores for normal calibration and validation data.

rolling_calibration_values = normal_calibration[feature_cols]
rolling_validation_values = normal_validation[feature_cols]

# Combine the normal periods in chronological order so that validation
# observations can use preceding calibration observations as history.
rolling_normal_values = pd.concat(
    [rolling_calibration_values, rolling_validation_values],
    axis=0,
)

# Shift by one observation to exclude the current reading from its reference.
rolling_reference = rolling_normal_values.shift(1)


rolling_mean = rolling_reference.rolling(
    window=ROLLING_WINDOW,
    min_periods=ROLLING_WINDOW,
).mean()

rolling_std = rolling_reference.rolling(
    window=ROLLING_WINDOW,
    min_periods=ROLLING_WINDOW,
).std(ddof=1)


# A zero standard deviation makes the Z-score undefined.
valid_std = rolling_std.where(rolling_std > 0)

rolling_z_scores = (
    rolling_normal_values.subtract(rolling_mean).abs().divide(valid_std)
)

# Split the calculated scores back into their original periods.
calibration_index = normal_calibration.index
validation_index = normal_validation.index

calibration_rolling_z_scores = rolling_z_scores.loc[calibration_index]
validation_rolling_z_scores = rolling_z_scores.loc[validation_index]

print("ROLLING Z-SCORE CALCULATION")
print("-" * 50)
print(f"Calibration observations: {len(calibration_rolling_z_scores):,}")
print(f"Validation observations: {len(validation_rolling_z_scores):,}")
print(f"Sensor features: {len(feature_cols)}")
print(f"Rolling window: {ROLLING_WINDOW}")
print(f"Z-score threshold: {ROLLING_Z_THRESHOLD}")
print(
    "Validation observations with at least one calculable score: "
    f"{validation_rolling_z_scores.notna().any(axis=1).sum():,}"
)
print(
    "Missing validation Z-scores: "
    f"{validation_rolling_z_scores.isna().sum().sum():,}"
)

ROLLING Z-SCORE CALCULATION
--------------------------------------------------
Calibration observations: 316,238
Validation observations: 79,060
Sensor features: 51
Rolling window: 60
Z-score threshold: 3.0
Validation observations with at least one calculable score: 79,060
Missing validation Z-scores: 2,053,532


In [15]:
# Diagnose missing rolling Z-scores in the normal validation period.

validation_score_missing = validation_rolling_z_scores.isna()

missing_scores_by_sensor = validation_score_missing.sum().sort_values(
    ascending=False
)

observations_without_any_score = validation_score_missing.all(axis=1).sum()

observations_with_all_scores = validation_score_missing.eq(False).all(axis=1).sum()

print("ROLLING Z-SCORE DIAGNOSTICS — NORMAL VALIDATION")
print("-" * 55)
print(f"Validation observations: {len(validation_rolling_z_scores):,}")
print(f"Sensor-level scores: {validation_rolling_z_scores.size:,}")
print(f"Missing sensor-level scores: {validation_score_missing.sum().sum():,}")
print(
    f"Missing-score percentage: "
    f"{validation_score_missing.sum().sum() / validation_rolling_z_scores.size:.2%}"
)
print(
    f"Observations with no calculable sensor scores: "
    f"{observations_without_any_score:,}"
)
print(
    f"Observations with scores for every sensor: "
    f"{observations_with_all_scores:,}"
)

print("\nSensors with the most missing scores:")
display(
    missing_scores_by_sensor.head(15).to_frame("missing_score_count")
)

ROLLING Z-SCORE DIAGNOSTICS — NORMAL VALIDATION
-------------------------------------------------------
Validation observations: 79,060
Sensor-level scores: 4,032,060
Missing sensor-level scores: 2,053,532
Missing-score percentage: 50.93%
Observations with no calculable sensor scores: 0
Observations with scores for every sensor: 0

Sensors with the most missing scores:


,missing_score_count
P202,79060
P206,79060
P204,79060
P601,79060
P603,79060
P502,79060
P403,79060
P402,79060
P401,79060
P404,79060


In [16]:
# Diagnose sensor variability and rolling-score availability.

sensor_diagnostics = pd.DataFrame(index=feature_cols)

sensor_diagnostics["calibration_unique_values"] = (
    normal_calibration[feature_cols].nunique(dropna=True)
)

sensor_diagnostics["validation_unique_values"] = (
    normal_validation[feature_cols].nunique(dropna=True)
)

sensor_diagnostics["calibration_std"] = (
    normal_calibration[feature_cols].std()
)

sensor_diagnostics["validation_std"] = (
    normal_validation[feature_cols].std()
)

sensor_diagnostics["validation_valid_scores"] = (
    validation_rolling_z_scores.notna().sum()
)

sensor_diagnostics["validation_missing_scores"] = (
    validation_rolling_z_scores.isna().sum()
)

sensor_diagnostics["calibration_constant"] = (
    sensor_diagnostics["calibration_unique_values"] <= 1
)

sensor_diagnostics["validation_constant"] = (
    sensor_diagnostics["validation_unique_values"] <= 1
)

print("SENSOR VARIABILITY AND ROLLING-SCORE DIAGNOSTICS")
print("-" * 60)

print(
    "Sensors constant during calibration: "
    f"{sensor_diagnostics['calibration_constant'].sum()}"
)

print(
    "Sensors constant during validation: "
    f"{sensor_diagnostics['validation_constant'].sum()}"
)

print(
    "Sensors with no valid validation Z-scores: "
    f"{sensor_diagnostics['validation_valid_scores'].eq(0).sum()}"
)

print("\nSensors with no valid validation Z-scores:")
display(
    sensor_diagnostics.loc[
        sensor_diagnostics["validation_valid_scores"].eq(0)
    ].sort_values(
        ["calibration_constant", "validation_constant"],
        ascending=False,
    )
)

print("\nSensors with the fewest valid validation Z-scores:")
display(
    sensor_diagnostics.sort_values(
        "validation_valid_scores",
        ascending=True,
    ).head(15)
)

SENSOR VARIABILITY AND ROLLING-SCORE DIAGNOSTICS
------------------------------------------------------------
Sensors constant during calibration: 9
Sensors constant during validation: 11
Sensors with no valid validation Z-scores: 11

Sensors with no valid validation Z-scores:


,calibration_unique_values,validation_unique_values,calibration_std,validation_std,validation_valid_scores,validation_missing_scores,calibration_constant,validation_constant
P202,1,1,0.000000,0.0,0,79060,True,True
P204,1,1,0.000000,0.0,0,79060,True,True
P206,1,1,0.000000,0.0,0,79060,True,True
P301,1,1,0.000000,0.0,0,79060,True,True
P401,1,1,0.000000,0.0,0,79060,True,True
P404,1,1,0.000000,0.0,0,79060,True,True
P502,1,1,0.000000,0.0,0,79060,True,True
P601,1,1,0.000000,0.0,0,79060,True,True
P603,1,1,0.000000,0.0,0,79060,True,True
P402,2,1,0.014335,0.0,0,79060,False,True



Sensors with the fewest valid validation Z-scores:


,calibration_unique_values,validation_unique_values,calibration_std,validation_std,validation_valid_scores,validation_missing_scores,calibration_constant,validation_constant
P204,1,1,0.000000,0.000000,0,79060,True,True
P206,1,1,0.000000,0.000000,0,79060,True,True
P202,1,1,0.000000,0.000000,0,79060,True,True
P301,1,1,0.000000,0.000000,0,79060,True,True
P402,2,1,0.014335,0.000000,0,79060,False,True
P403,2,1,0.013773,0.000000,0,79060,False,True
P401,1,1,0.000000,0.000000,0,79060,True,True
P404,1,1,0.000000,0.000000,0,79060,True,True
P601,1,1,0.000000,0.000000,0,79060,True,True
P603,1,1,0.000000,0.000000,0,79060,True,True


In [17]:
# Apply the rolling Z-score threshold to normal validation observations.

validation_rolling_flags = (
    validation_rolling_z_scores > ROLLING_Z_THRESHOLD
)

# Count sensors flagged and sensors with calculable scores per observation.
validation_rolling_anomaly_score = validation_rolling_flags.sum(axis=1)

validation_rolling_scorable_count = (
    validation_rolling_z_scores.notna().sum(axis=1)
)

# An observation is flagged if at least one scorable sensor exceeds the threshold.
validation_rolling_anomaly = validation_rolling_anomaly_score > 0

print("ROLLING STATISTICAL DETECTION — NORMAL VALIDATION")
print("-" * 55)

print(f"Validation observations: {len(validation_rolling_z_scores):,}")

print(
    "Observations flagged as anomalous: "
    f"{validation_rolling_anomaly.sum():,}"
)

print(
    "Validation anomaly rate: "
    f"{validation_rolling_anomaly.mean():.4%}"
)

print(
    "Minimum scorable sensors per observation: "
    f"{validation_rolling_scorable_count.min()}"
)

print(
    "Maximum scorable sensors per observation: "
    f"{validation_rolling_scorable_count.max()}"
)

print(
    "Mean scorable sensors per observation: "
    f"{validation_rolling_scorable_count.mean():.2f}"
)

print(
    "Mean flagged sensors per observation: "
    f"{validation_rolling_anomaly_score.mean():.4f}"
)

print("\nSensors with the most rolling-threshold violations:")

display(
    validation_rolling_flags.sum()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("violation_count")
)

ROLLING STATISTICAL DETECTION — NORMAL VALIDATION
-------------------------------------------------------
Validation observations: 79,060
Observations flagged as anomalous: 33,506
Validation anomaly rate: 42.3805%
Minimum scorable sensors per observation: 21
Maximum scorable sensors per observation: 35
Mean scorable sensors per observation: 25.03
Mean flagged sensors per observation: 0.6232

Sensors with the most rolling-threshold violations:


,violation_count
AIT203,3911
AIT501,3455
AIT402,3128
AIT504,3126
AIT502,2744
PIT502,2743
AIT202,2628
AIT503,2419
FIT504,2251
AIT201,2248


### 11.5 Rolling Statistical Detection — Validation Findings

The rolling statistical baseline used a window of 60 preceding observations and an absolute Z-score threshold of 3.0. Rolling reference statistics excluded the observation being assessed, and the normal validation period was evaluated chronologically using preceding normal observations as history.

The evaluation produced the following results:

- Validation observations: 79,060
- Observations flagged as anomalous: 33,506
- Validation anomaly rate: 42.3805%
- Mean number of scorable sensors per observation: 25.03 out of 51
- Mean number of sensors exceeding the threshold per observation: 0.6232

AIT203 had the highest number of rolling-threshold violations (3,911), followed by AIT501 (3,455), AIT402 (3,128), AIT504 (3,126), and AIT502 (2,744).

#### Interpretation and Limitations

The rolling baseline flagged a smaller proportion of normal validation observations than the rule-based baseline. However, this difference alone does not establish that the rolling method is superior.

Only an average of 25.03 of the 51 sensors had calculable rolling Z-scores per observation. Eleven sensors had no valid rolling Z-scores throughout validation because their validation values were constant. Other sensors had intermittent missing scores, including cases where their preceding rolling windows had zero variance.

Consequently, the detector's decisions were based on the subset of sensors with valid scores at each observation. The results should be interpreted with this coverage limitation in mind.

The 3.0 threshold and 60-observation window were used as initial statistical baseline settings, not as verified SWaT operating limits. No attack labels were used to select these settings.

The 42.3805% validation anomaly rate indicates that this baseline frequently flags observations from the available normal validation period. The cause of these flags has not yet been established, and the results do not demonstrate confirmed attacks.

## 12. Baseline 3 — Simple Statistical Threshold

### 12.1 Objective

The simple statistical threshold baseline identifies observations that deviate substantially from the statistical behaviour observed during the normal calibration period.

Unlike rolling statistical detection, which updates its reference using preceding observations, this method uses a fixed reference calculated from the calibration data. This provides a straightforward comparison for assessing whether a fixed statistical reference can identify unusual sensor readings in the later normal validation period.

### 12.2 Statistical Method

For each sensor, calculate the mean and standard deviation using the normal calibration observations.

Calibration mean:

μⱼ = (1 / n) × Σ xᵢ,ⱼ

Calibration standard deviation:

σⱼ = √[Σ(xᵢ,ⱼ − μⱼ)² / (n − 1)]

where n is the number of calibration observations and j identifies a sensor.

For each validation observation, calculate the absolute standardized deviation:

zₜ,ⱼ = |xₜ,ⱼ − μⱼ| / σⱼ

An observation is flagged as anomalous when at least one sensor has an absolute standardized deviation greater than the predefined threshold.

### 12.3 Implementation Principles

- Calculate the reference mean and standard deviation from normal calibration data only.
- Apply the fixed reference to normal validation observations without updating it using validation values.
- Preserve chronological order.
- Handle sensors with zero calibration standard deviation explicitly.
- Do not use attack labels to select thresholds.
- Report the validation anomaly rate, sensor-level violations, and the number of sensors that can be scored.
- Keep this baseline distinct from the rolling statistical method by using fixed calibration statistics.

### 12.4 Interpretation

A fixed statistical reference provides a simple benchmark for identifying deviations from calibration-period behaviour. However, it may flag observations when normal sensor behaviour changes over time.

The results will be interpreted alongside those of the rule-based and rolling statistical baselines. A high validation anomaly rate indicates frequent detections in the available normal validation period, but does not by itself establish the cause of those detections or demonstrate confirmed attacks.

In [18]:
# Calculate fixed statistical references from normal calibration data.

simple_threshold_mean = normal_calibration[feature_cols].mean()
simple_threshold_std = normal_calibration[feature_cols].std(ddof=1)

simple_threshold_reference = pd.DataFrame({
    "calibration_mean": simple_threshold_mean,
    "calibration_std": simple_threshold_std,
})

print("SIMPLE STATISTICAL THRESHOLD — CALIBRATION REFERENCE")
print("-" * 60)
print(f"Calibration observations: {len(normal_calibration):,}")
print(f"Sensor features: {len(feature_cols)}")
print(
    "Sensors with zero calibration standard deviation: "
    f"{simple_threshold_std.eq(0).sum()}"
)
print(
    "Sensors with non-zero calibration standard deviation: "
    f"{simple_threshold_std.gt(0).sum()}"
)

print("\nCalibration reference statistics:")
display(simple_threshold_reference)

SIMPLE STATISTICAL THRESHOLD — CALIBRATION REFERENCE
------------------------------------------------------------
Calibration observations: 316,238
Sensor features: 51
Sensors with zero calibration standard deviation: 9
Sensors with non-zero calibration standard deviation: 42

Calibration reference statistics:


,calibration_mean,calibration_std
FIT101,1.842657,1.135979
LIT101,588.903952,115.436900
MV101,1.715072,0.463117
P101,1.750536,0.432703
P102,1.000183,0.013542
AIT201,220.919175,34.202924
AIT202,8.498531,0.098962
AIT203,322.458188,12.264168
FIT201,1.838846,1.058179
MV201,1.747159,0.444177


In [19]:
# Apply the fixed calibration reference to normal validation observations.

validation_values = normal_validation[feature_cols]

# Calculate absolute standardized deviations using calibration statistics.
simple_threshold_valid_std = simple_threshold_std.where(
    simple_threshold_std > 0
)

simple_threshold_z_scores = (
    validation_values
    .subtract(simple_threshold_mean)
    .abs()
    .divide(simple_threshold_valid_std)
)

# Flag sensor-level deviations exceeding the fixed threshold.
SIMPLE_Z_THRESHOLD = 3.0

simple_threshold_flags = (
    simple_threshold_z_scores > SIMPLE_Z_THRESHOLD
)

# Summarize detections for each validation observation.
simple_threshold_anomaly_score = simple_threshold_flags.sum(axis=1)
simple_threshold_anomaly = simple_threshold_anomaly_score > 0

simple_threshold_scorable_count = (
    simple_threshold_z_scores.notna().sum(axis=1)
)

print("SIMPLE STATISTICAL THRESHOLD — NORMAL VALIDATION")
print("-" * 60)
print(f"Validation observations: {len(validation_values):,}")
print(f"Z-score threshold: {SIMPLE_Z_THRESHOLD:.1f}")
print(
    f"Observations flagged as anomalous: "
    f"{simple_threshold_anomaly.sum():,}"
)
print(
    f"Validation anomaly rate: "
    f"{simple_threshold_anomaly.mean():.4%}"
)
print(
    f"Minimum scorable sensors per observation: "
    f"{simple_threshold_scorable_count.min()}"
)
print(
    f"Maximum scorable sensors per observation: "
    f"{simple_threshold_scorable_count.max()}"
)
print(
    f"Mean scorable sensors per observation: "
    f"{simple_threshold_scorable_count.mean():.2f}"
)
print(
    f"Mean flagged sensors per observation: "
    f"{simple_threshold_anomaly_score.mean():.4f}"
)

print("\nSensors with the most threshold violations:")
display(
    simple_threshold_flags.sum()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("violation_count")
)

SIMPLE STATISTICAL THRESHOLD — NORMAL VALIDATION
------------------------------------------------------------
Validation observations: 79,060
Z-score threshold: 3.0
Observations flagged as anomalous: 10,852
Validation anomaly rate: 13.7263%
Minimum scorable sensors per observation: 42
Maximum scorable sensors per observation: 42
Mean scorable sensors per observation: 42.00
Mean flagged sensors per observation: 0.2273

Sensors with the most threshold violations:


,violation_count
AIT401,4410
MV304,3722
MV303,3318
MV301,1413
FIT601,1037
P602,921
MV302,593
MV101,453
MV201,411
AIT502,311


### 12.5 Simple Statistical Threshold — Validation Findings

The simple statistical threshold baseline used the mean and sample standard deviation of each sensor during the normal calibration period. These fixed statistics were applied to normal validation observations using an absolute Z-score threshold of 3.0.

The evaluation produced the following results:

- Validation observations: 79,060
- Observations flagged as anomalous: 10,852
- Validation anomaly rate: 13.7263%
- Sensors with calculable scores per observation: 42 out of 51
- Mean number of sensors exceeding the threshold per observation: 0.2273

AIT401 had the highest number of threshold violations (4,410), followed by MV304 (3,722), MV303 (3,318), MV301 (1,413), and FIT601 (1,037).

#### Interpretation and Limitations

The fixed statistical baseline flagged a smaller proportion of normal validation observations than the rule-based and rolling statistical baselines. However, anomaly rate alone does not establish comparative superiority.

Nine sensors had zero standard deviation during calibration and could not produce conventional Z-scores. The detector therefore evaluated each observation using 42 sensors. Unlike the rolling baseline, this sensor coverage was consistent across the validation period.

The reference statistics were calculated exclusively from normal calibration data. Validation observations were evaluated against this fixed reference, and attack labels were not used to select the threshold.

The 13.7263% validation anomaly rate indicates that the baseline frequently flags observations from the available normal validation period. The cause of these detections has not been established, and they should not be interpreted as confirmed attacks.

## 13. Comparison of Baseline Results

The three baseline methods were evaluated on the same 79,060 observations in the normal validation period. Each method used parameters established without using attack labels.

### 13.1 Comparison Table

| Baseline Method | Observations Flagged | Validation Anomaly Rate | Mean Scorable Sensors per Observation | Mean Flagged Sensors per Observation |
|:--|--:|--:|--:|--:|
| Rule-Based Detection | 73,295 | 92.7081% | Not separately calculated | 1.9914 |
| Rolling Statistical Detection | 33,506 | 42.3805% | 25.03 | 0.6232 |
| Simple Statistical Threshold | 10,852 | 13.7263% | 42 | 0.2273 |

### 13.2 Interpretation

The three methods produced different detection rates on the normal validation period.

The rule-based detector flagged the largest proportion of observations. The rolling statistical detector flagged fewer observations, while the simple statistical threshold produced the lowest detection rate.

These differences reflect the methods' distinct scoring procedures and reference statistics. They do not establish that one method is superior to another.

The rolling statistical detector could not calculate valid Z-scores for every sensor because some rolling windows had zero variance. Eleven sensors had no valid rolling Z-scores throughout the validation period. Other sensors had intermittent missing scores. The simple statistical threshold consistently evaluated 42 of the 51 sensors because nine sensors had zero standard deviation during calibration.

### 13.3 Limitations

These results describe detection behaviour on the normal validation data. They do not establish how effectively any baseline detects attack-labelled observations.

The observed validation anomaly rates should not automatically be interpreted as definitive false-positive rates. Although the observations are labelled normal in the dataset, the analysis has not independently established the cause of every flagged observation.

No thresholds were adjusted using attack labels, and no claim of superior detection performance is made at this stage.

The next evaluation stage must examine both normal and attack-labelled observations using a consistent evaluation procedure. This will allow us to calculate the confusion matrix, including true positives, true negatives, false positives, and false negatives, and derive appropriate classification metrics.

## 14. Conclusion

This notebook established three baseline anomaly-detection methods using the available SWaT normal data: rule-based detection, rolling statistical detection, and a simple statistical threshold.

The analysis used 395,298 eligible normal observations with valid timestamps and complete sensor values. These observations were divided chronologically into 316,238 calibration observations and 79,060 validation observations. The remaining normal observations were excluded from this baseline analysis because of missing sensor values or invalid timestamps; the missing-value block also contains the duplicated normal-data segment identified in Notebook 02, which is therefore excluded from this analysis as well.

### 14.1 Detection Rates on Normal Validation Data

| Baseline | Observations Flagged | Normal-Validation Anomaly Rate |
|:--|--:|--:|
| Rule-Based Detection | 73,295 | 92.7081% |
| Rolling Statistical Detection | 33,506 | 42.3805% |
| Simple Statistical Threshold | 10,852 | 13.7263% |

These results establish a reference for subsequent comparisons but do not demonstrate which method is most effective at detecting attacks.

### 14.2 Sensor Coverage

The analysis also identified limitations in sensor-level scoring:

- **Rule-based:** all 51 sensors evaluated; mean 1.9914 violated sensors per observation.
- **Rolling statistical:** mean 25.03 of 51 sensors scorable per observation; 11 sensors had no valid rolling Z-scores throughout the validation period (constant during validation), and additional sensors had intermittent missing scores when the preceding rolling windows had zero variance.
- **Simple statistical:** 42 of 51 sensors scorable per observation (nine sensors had zero standard deviation during calibration).

### 14.3 Scope and Interpretation

The empirical limits and statistical thresholds were derived from normal calibration data. Attack labels were not used to construct the baselines or select their thresholds. No engineering safety limits were invented, and flagged observations have not been treated as confirmed cyberattacks.

Normal-validation anomaly rates should not be interpreted as definitive false-positive rates. Although the observations are labelled normal, the analysis has not independently established the cause of every flagged observation.

### 14.4 Next Steps

Attack-labelled evaluation is outside the scope of this notebook (Section 6) and is deferred to the next stage. Subsequent evaluation will examine normal and attack-labelled observations, calculate confusion-matrix counts, and report appropriate performance metrics. Error and event analysis will then investigate false positives, false negatives, and the temporal behaviour of detections.